# Getting Started with LLM APIs: OpenAI, Anthropic & Google

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

Before building agents and advanced workflows, you need to know how to **talk to a
Large Language Model programmatically**. This notebook walks you through setting up
and using the APIs from the three major LLM providers:

| Provider | Model Family | Python SDK |
|----------|-------------|------------|
| **OpenAI** | GPT-4o, o3, o4-mini | `openai` |
| **Anthropic** | Claude (Sonnet, Opus, Haiku) | `anthropic` |
| **Google** | Gemini (Pro, Flash) | `google-genai` |

## What You'll Learn

1. How to get an API key from each provider
2. How to install and configure each SDK
3. How to send your first message and get a response
4. How to use **system prompts** to shape the model's behavior
5. How to do **multi-turn conversations**
6. Side-by-side comparison of all three APIs

## Prerequisites

- A Google Colab environment (or local Python 3.9+)
- At least one API key from the providers above (free tiers are available)

---
## Step 1: Install the SDKs

In [ ]:
# Install all three SDKs
# !pip install openai anthropic google-genai

---
## Step 2: Set Up Your API Keys

Each provider requires an API key. Here's where to get them:

| Provider | Where to get your key | Free tier? |
|----------|----------------------|------------|
| **OpenAI** | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) | Pay-as-you-go (small free credit for new accounts) |
| **Anthropic** | [console.anthropic.com/settings/keys](https://console.anthropic.com/settings/keys) | Pay-as-you-go (small free credit for new accounts) |
| **Google** | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) | Free tier available |

> **Security tip:** Never hardcode API keys in notebooks. We use `getpass` to enter
> them interactively and `os.environ` to store them for the session.

In [ ]:
import os
import getpass

def setup_key(env_var, provider_name):
    """Prompt for an API key if not already set."""
    if env_var not in os.environ:
        key = getpass.getpass(f"Enter your {provider_name} API key (or press Enter to skip): ")
        if key.strip():
            os.environ[env_var] = key.strip()
            print(f"{provider_name} key configured.")
        else:
            print(f"{provider_name} skipped.")
    else:
        print(f"{provider_name} key already set.")

setup_key("OPENAI_API_KEY", "OpenAI")
setup_key("ANTHROPIC_API_KEY", "Anthropic")
setup_key("GOOGLE_API_KEY", "Google")

---
## OpenAI (GPT-4o, o3, o4-mini)

OpenAI's API uses a **messages** format where you send a list of messages
with roles (`system`, `user`, `assistant`) and get back a completion.

### Basic Usage

In [ ]:
from openai import OpenAI

openai_client = OpenAI()  # reads OPENAI_API_KEY from environment

# Send a simple message
response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What is X-ray diffraction in one sentence?"}
    ],
)

print(response.choices[0].message.content)

### With a System Prompt

The **system prompt** sets the model's persona and instructions. It's sent as
the first message with `role: "system"`.

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a materials science expert. Give concise, "
                "technically precise answers. Use proper notation for "
                "crystallographic directions and planes."
            ),
        },
        {
            "role": "user",
            "content": (
                "What are the Miller indices of the most intense "
                "diffraction peak for FCC copper using Cu Ka radiation?"
            ),
        },
    ],
)

print(response.choices[0].message.content)

### Multi-Turn Conversation

To have a back-and-forth conversation, you maintain a list of messages and
append each new exchange.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful materials science tutor.",
    },
    {
        "role": "user",
        "content": "What is a perovskite crystal structure?",
    },
]

# First turn
response = openai_client.chat.completions.create(model="gpt-4o", messages=messages)
assistant_reply = response.choices[0].message.content
print("Assistant:", assistant_reply)
print()

# Add the reply and ask a follow-up
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "Give me three examples of perovskite materials and their applications."})

response = openai_client.chat.completions.create(model="gpt-4o", messages=messages)
print("Assistant:", response.choices[0].message.content)

---
## Anthropic (Claude Sonnet, Opus, Haiku)

Anthropic's API is similar to OpenAI's but with a few differences:
- The **system prompt** is a separate parameter, not a message
- The model name format is different (e.g., `claude-sonnet-4-6`)
- Responses use `.content[0].text` instead of `.choices[0].message.content`

### Basic Usage

In [ ]:
import anthropic

anthropic_client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is X-ray diffraction in one sentence?"}
    ],
)

print(response.content[0].text)

### With a System Prompt

In the Anthropic API, the system prompt is a **separate parameter**, not a message
in the list. This is a key difference from OpenAI.

In [ ]:
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system=(
        "You are a materials science expert. Give concise, "
        "technically precise answers. Use proper notation for "
        "crystallographic directions and planes."
    ),
    messages=[
        {
            "role": "user",
            "content": (
                "What are the Miller indices of the most intense "
                "diffraction peak for FCC copper using Cu Ka radiation?"
            ),
        },
    ],
)

print(response.content[0].text)

### Multi-Turn Conversation

In [ ]:
messages = [
    {"role": "user", "content": "What is a perovskite crystal structure?"},
]

# First turn
response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are a helpful materials science tutor.",
    messages=messages,
)
assistant_reply = response.content[0].text
print("Assistant:", assistant_reply)
print()

# Add reply and follow up
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "Give me three examples of perovskite materials and their applications."})

response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are a helpful materials science tutor.",
    messages=messages,
)
print("Assistant:", response.content[0].text)

---
## Google (Gemini Pro, Flash)

Google's Gemini API has its own style:
- Uses the `google-genai` SDK
- Creates a **client** and calls `generate_content`
- System instructions are passed via a `config` parameter
- Multi-turn uses a built-in **chat** interface

### Basic Usage

In [ ]:
from google import genai

google_client = genai.Client()  # reads GOOGLE_API_KEY from environment

response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What is X-ray diffraction in one sentence?",
)

print(response.text)

### With a System Prompt

Google calls the system prompt **system instruction** and passes it
via a config object.

In [ ]:
from google.genai import types

response = google_client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a materials science expert. Give concise, "
            "technically precise answers. Use proper notation for "
            "crystallographic directions and planes."
        ),
    ),
    contents=(
        "What are the Miller indices of the most intense "
        "diffraction peak for FCC copper using Cu Ka radiation?"
    ),
)

print(response.text)

### Multi-Turn Conversation

Google provides a built-in `chats.create` interface that manages conversation
history for you.

In [ ]:
chat = google_client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a helpful materials science tutor.",
    ),
)

# First turn
response = chat.send_message("What is a perovskite crystal structure?")
print("Assistant:", response.text)
print()

# Follow-up (history is tracked automatically)
response = chat.send_message("Give me three examples of perovskite materials and their applications.")
print("Assistant:", response.text)

---
## Side-by-Side Comparison

Let's send the **same prompt** to all three models and compare their responses.

In [ ]:
PROMPT = (
    "Explain the difference between XRD and XRF in 2-3 sentences. "
    "Focus on what each technique measures and when you would use each one."
)

results = {}

# OpenAI
if os.environ.get("OPENAI_API_KEY"):
    r = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": PROMPT}],
    )
    results["OpenAI (GPT-4o)"] = r.choices[0].message.content

# Anthropic
if os.environ.get("ANTHROPIC_API_KEY"):
    r = anthropic_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[{"role": "user", "content": PROMPT}],
    )
    results["Anthropic (Claude Sonnet)"] = r.content[0].text

# Google
if os.environ.get("GOOGLE_API_KEY"):
    r = google_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=PROMPT,
    )
    results["Google (Gemini Flash)"] = r.text

for provider, answer in results.items():
    print(f"{'='*60}")
    print(f"{provider}")
    print(f"{'='*60}")
    print(answer)
    print()

---
## Quick Reference Card

### Installation
```bash
pip install openai anthropic google-genai
```

### API Comparison

| | OpenAI | Anthropic | Google |
|---|--------|-----------|--------|
| **SDK** | `openai` | `anthropic` | `google-genai` |
| **Env variable** | `OPENAI_API_KEY` | `ANTHROPIC_API_KEY` | `GOOGLE_API_KEY` |
| **Client** | `OpenAI()` | `anthropic.Anthropic()` | `genai.Client()` |
| **Method** | `chat.completions.create()` | `messages.create()` | `models.generate_content()` |
| **System prompt** | `role: "system"` message | `system=` parameter | `config=GenerateContentConfig(system_instruction=)` |
| **Get response text** | `.choices[0].message.content` | `.content[0].text` | `.text` |
| **Multi-turn** | Maintain message list | Maintain message list | `chats.create()` + `send_message()` |
| **Model names** | `gpt-4o`, `o3`, `o4-mini` | `claude-sonnet-4-6`, `claude-haiku-4-5-20251001` | `gemini-2.5-flash`, `gemini-2.5-pro` |

### Code Templates

**OpenAI:**
```python
from openai import OpenAI
client = OpenAI()
r = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "system prompt here"},
        {"role": "user", "content": "your question"},
    ],
)
print(r.choices[0].message.content)
```

**Anthropic:**
```python
import anthropic
client = anthropic.Anthropic()
r = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="system prompt here",
    messages=[{"role": "user", "content": "your question"}],
)
print(r.content[0].text)
```

**Google:**
```python
from google import genai
from google.genai import types
client = genai.Client()
r = client.models.generate_content(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(system_instruction="system prompt here"),
    contents="your question",
)
print(r.text)
```

---
## Next Steps

Now that you can talk to LLMs programmatically, you're ready to:

- **`01_prompt_engineering.ipynb`** — prompt engineering strategies for materials science
- **`02_structured_outputs.ipynb`** — get parseable JSON out of an LLM
- **`03_rag_literature.ipynb`** — ground answers in your own document corpus (RAG)
- **`04_agent_microscopy.ipynb`** — build a scientific agent with tool use, end-to-end
- **`05_mcp_server.ipynb`** — expose tools via the Model Context Protocol
- **`06_multi_agent_eval_caching.ipynb`** — orchestration, evaluation, prompt caching
- **`07_langgraph_agent.ipynb`** — the same agent rebuilt as a LangGraph state machine
- **`08_openalex_papers.ipynb`** — fetch real scientific papers via the free OpenAlex API and feed them to an agent

The remaining notebooks all use the Anthropic (Claude) API, but the concepts apply to all providers.